In [ ]:
!pip install -q -U datasets huggingface_hub requests

import json
import os
import time
import random
import traceback
from datetime import datetime, timezone

import requests
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi

In [2]:
# ## 1. CONFIG — edit everything in this cell


HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")


SYSTEM_LABEL = "finetuned"          

# INPUT_DATASET_REPO = "businessrules/Qwen_base_tuned_exp10_results"  
# OUTPUT_DATASET_REPO = "businessrules/exp10_promptA_results" 
INPUT_DATASET_REPO = "businessrules/GPT4_baseline_results"   
OUTPUT_DATASET_REPO = "businessrules/gpt4.1_promptA_results"  


ID_COLUMN = "id"                   
# OUTPUT_COLUMN = "finetuned_model_prediction" 
OUTPUT_COLUMN = "gpt4_prediction"
MAX_ITEMS = None
INPUT_SPLIT = "train" 
# ---- OpenRouter ----
OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")

JUDGE_MODELS = [
    "openai/gpt-4o-mini",
    "anthropic/claude-sonnet-4.6",
    "google/gemini-2.5-flash",
]

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
REQUEST_TIMEOUT = 120
TEMPERATURE = 0.0
MAX_TOKENS = 3000

# ---- Checkpointing ----
SAVE_EVERY = 10          # push to the HF output dataset after this many NEW results
MAX_RETRIES = 5          # per API call, with exponential backoff
BASE_BACKOFF_SECONDS = 5



In [3]:
# ## 2. Prompt A — system prompt and user prompt template
SYSTEM_PROMPT = """You are a strict technical documentation auditor. You evaluate the STRUCTURE and
PRESENTATION QUALITY of a list of business rules extracted from source code. You are
NOT evaluating whether the rules are factually correct, complete, or non-hallucinated —
ignore correctness entirely. You are only judging how the output is written and formatted.

You will not see the source code or any reference/gold answer. Judge the text purely on
its own presentational merits.

Be skeptical by default. Assume a defect is present until you can point to specific
evidence in the text that it is not. Do not round up. A score of 5 requires zero
exceptions found during your checklist review — if you find even one instance of a
problem, the score for that dimension must reflect it.

You must complete your evidence checklist BEFORE assigning any numeric score. Do not
decide the score first and rationalize it afterward. The score must be a deterministic
function of the checklist answers you give (formulas provided per dimension below).

Output valid JSON only. No prose outside the JSON object. No markdown code fences."""

USER_PROMPT_TEMPLATE = """Evaluate the following generated output on structural and presentational quality only.

GENERATED OUTPUT:
\"\"\"
{generated_output}
\"\"\"

Work through each dimension below in order. For each dimension, first fill in the
checklist/evidence fields, then compute the score using the given formula. Do not skip
the evidence fields.

---

### Dimension 1: Title presence
- has_title: true/false — does the output begin with a clear title or heading that names
  what the document is (e.g. "# Business Rules for X")? A generic first line of body text
  does not count as a title.
- This is binary. No score to compute; report has_title only.

### Dimension 2: Markdown formatting quality
Check each of the following independently and record true/false with a one-sentence
reason for each:
- uses_headings: proper markdown heading syntax used for at least the title and one
  section (not just bold text pretending to be a heading)
- uses_sections_or_subheadings: rules are grouped under at least one meaningful section
  or category heading, not dumped as one flat list
- rules_numbered_or_bulleted: each individual rule is a distinct numbered or bulleted
  list item, not run together in paragraph prose
- grouped_by_criteria: rules are grouped by a sensible criterion (e.g. by feature, by
  entity, by workflow stage) rather than in arbitrary/random order
- consistent_styling: consistent use of bold/italic/code formatting for emphasis
  throughout (not inconsistent — e.g. some rules styled, others not)
- no_broken_markdown: no malformed markdown syntax (unclosed bold markers, broken list
  numbering, headings missing space after #, etc.)

formatting_checks_passed = count of true values above (integer 0-6)
formatting_score (1-5) =
  1 if formatting_checks_passed <= 1
  2 if formatting_checks_passed == 2
  3 if formatting_checks_passed == 3 or 4
  4 if formatting_checks_passed == 5
  5 if formatting_checks_passed == 6

### Dimension 3: Readability of individual rules
For EACH rule in the output (list them by number), record:
- rule_id
- uses_conditional_language: true/false — does it explicitly use conditional structure
  (if/when/unless/otherwise) where the rule logic requires a condition? (If the rule is
  unconditional by nature, mark true — not applicable is not a penalty.)
- uses_precise_quantifiers: true/false — where the rule involves a collection or multiple
  items, does it use precise quantifier language (each/all/any/none/at least one) rather
  than vague plural phrasing?
- unambiguous: true/false — is the rule free of ambiguous pronouns or dangling
  references (e.g. "it must be validated" without saying what "it" is)?
- self_contained: true/false — can this rule be understood without needing to read
  other rules first?

readable_rules_count = number of rules where ALL FOUR fields above are true
total_rules_count = total number of rules evaluated
readability_ratio = readable_rules_count / total_rules_count (0 if total_rules_count is 0)
readability_score (1-5) =
  1 if readability_ratio < 0.2
  2 if readability_ratio < 0.4
  3 if readability_ratio < 0.6
  4 if readability_ratio < 0.85
  5 if readability_ratio >= 0.85

---

Calibration reference (for formatting_score and readability_score):
- A score of 5 means you found literally zero instances of the described problem.
- A score of 3 is a genuinely mixed/mediocre document — do not use 3 as a default
  "safe middle" score; only assign it when the checklist math produces it.
- A score of 1-2 is common and expected for early-stage or poorly-formatted model
  output — do not be reluctant to assign low scores.

Return ONLY a JSON object matching this exact schema (see JSON_OUTPUT_SCHEMA below).
Every rule listed in the "rules" array under readability must correspond to a rule you
can identify in the generated output — do not invent or omit rules.

JSON_OUTPUT_SCHEMA:
{{
  "title": {{
    "has_title": true,
    "evidence": "First line is '# Business Rules: Booking Cancellation Policy'"
  }},
  "formatting": {{
    "uses_headings": {{ "value": true, "reason": "..." }},
    "uses_sections_or_subheadings": {{ "value": true, "reason": "..." }},
    "rules_numbered_or_bulleted": {{ "value": true, "reason": "..." }},
    "grouped_by_criteria": {{ "value": false, "reason": "..." }},
    "consistent_styling": {{ "value": true, "reason": "..." }},
    "no_broken_markdown": {{ "value": true, "reason": "..." }},
    "formatting_checks_passed": 5,
    "formatting_score": 4
  }},
  "readability": {{
    "rules": [
      {{
        "rule_id": 1,
        "uses_conditional_language": true,
        "uses_precise_quantifiers": true,
        "unambiguous": true,
        "self_contained": true
      }}
    ],
    "total_rules_count": 1,
    "readable_rules_count": 1,
    "readability_ratio": 1.0,
    "readability_score": 5
  }},
  "overall_notes": "Free-text summary of the single biggest structural weakness, 1-2 sentences max."
}}"""

In [ ]:
# ## 3. Load the input dataset (test split)
print(f"Loading {INPUT_DATASET_REPO} split={INPUT_SPLIT} (system={SYSTEM_LABEL}) ...")
input_ds = load_dataset(INPUT_DATASET_REPO, split=INPUT_SPLIT, token=HF_TOKEN)
print(f"Loaded {len(input_ds)} rows. Columns: {input_ds.column_names}")

if ID_COLUMN not in input_ds.column_names:
    print(f"WARNING: id_column '{ID_COLUMN}' not found — falling back to row index as id.")
if OUTPUT_COLUMN not in input_ds.column_names:
    raise ValueError(
        f"OUTPUT_COLUMN '{OUTPUT_COLUMN}' not found in {INPUT_DATASET_REPO}. "
        f"Available columns: {input_ds.column_names}"
    )
if MAX_ITEMS is not None:
    input_ds = input_ds.select(range(min(MAX_ITEMS, len(input_ds))))
    print(f"MAX_ITEMS set — using only the first {len(input_ds)} rows for this run.")


# ## 4. OpenRouter call + JSON parsing/validation helpers
def call_openrouter(model: str, system_prompt: str, user_prompt: str) -> str:
    """Calls OpenRouter chat completions, returns raw text content. Retries with
    exponential backoff on transient failures (429, 5xx, timeouts)."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "response_format": {"type": "json_object"},
    }

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(
                OPENROUTER_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT
            )
            if resp.status_code == 429 or resp.status_code >= 500:
                raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")
            resp.raise_for_status()
            data = resp.json()
            return data["choices"][0]["message"]["content"]
        except Exception as e:
            last_err = e
            sleep_s = BASE_BACKOFF_SECONDS * (2 ** (attempt - 1)) + random.uniform(0, 2)
            print(f"  [retry {attempt}/{MAX_RETRIES}] {model} call failed: {e}. "
                  f"Sleeping {sleep_s:.1f}s ...")
            time.sleep(sleep_s)
    raise RuntimeError(f"OpenRouter call failed after {MAX_RETRIES} retries: {last_err}")


def strip_code_fences(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1] if "\n" in t else t
        if t.endswith("```"):
            t = t.rsplit("```", 1)[0]
    return t.strip()


def recompute_checks(parsed: dict) -> dict:
    """Recomputes formatting_checks_passed / formatting_score and
    readability_ratio / readability_score from the boolean fields, per the
    prompt's own instructions. Flags mismatches vs what the model reported."""
    issues = []

    fmt = parsed.get("formatting", {})
    fmt_keys = [
        "uses_headings", "uses_sections_or_subheadings", "rules_numbered_or_bulleted",
        "grouped_by_criteria", "consistent_styling", "no_broken_markdown",
    ]
    try:
        passed = sum(1 for k in fmt_keys if fmt.get(k, {}).get("value") is True)
    except AttributeError:
        passed = None
        issues.append("formatting fields malformed")

    def fmt_score_from_count(n):
        if n is None:
            return None
        if n <= 1:
            return 1
        if n == 2:
            return 2
        if n in (3, 4):
            return 3
        if n == 5:
            return 4
        return 5

    recomputed_fmt_score = fmt_score_from_count(passed)
    if passed is not None and fmt.get("formatting_checks_passed") != passed:
        issues.append(
            f"formatting_checks_passed mismatch: model said "
            f"{fmt.get('formatting_checks_passed')}, recomputed {passed}"
        )
    if (recomputed_fmt_score is not None
            and fmt.get("formatting_score") != recomputed_fmt_score):
        issues.append(
            f"formatting_score mismatch: model said {fmt.get('formatting_score')}, "
            f"recomputed {recomputed_fmt_score}"
        )
    if fmt.get("formatting_score") == 5 and fmt.get("grouped_by_criteria", {}).get("value") is False:
        issues.append("CONTRADICTION: formatting_score=5 but grouped_by_criteria=false")

    read = parsed.get("readability", {})
    rules = read.get("rules", [])
    total = len(rules)
    readable = sum(
        1 for r in rules
        if r.get("uses_conditional_language") is True
        and r.get("uses_precise_quantifiers") is True
        and r.get("unambiguous") is True
        and r.get("self_contained") is True
    )
    ratio = (readable / total) if total > 0 else 0.0

    def read_score_from_ratio(r):
        if r < 0.2:
            return 1
        if r < 0.4:
            return 2
        if r < 0.6:
            return 3
        if r < 0.85:
            return 4
        return 5

    recomputed_read_score = read_score_from_ratio(ratio)
    if read.get("total_rules_count") != total:
        issues.append(
            f"total_rules_count mismatch: model said {read.get('total_rules_count')}, "
            f"recomputed {total}"
        )
    if read.get("readable_rules_count") != readable:
        issues.append(
            f"readable_rules_count mismatch: model said {read.get('readable_rules_count')}, "
            f"recomputed {readable}"
        )
    if read.get("readability_score") != recomputed_read_score:
        issues.append(
            f"readability_score mismatch: model said {read.get('readability_score')}, "
            f"recomputed {recomputed_read_score}"
        )

    return {
        "recomputed_formatting_checks_passed": passed,
        "recomputed_formatting_score": recomputed_fmt_score,
        "recomputed_readability_ratio": ratio,
        "recomputed_readability_score": recomputed_read_score,
        "validation_issues": issues,
        "is_consistent": len(issues) == 0,
    }


def judge_one(model: str, generated_output: str) -> dict:
    """Runs Prompt A once and returns a flat result dict (never raises — errors are
    captured in the row so the pipeline keeps going)."""
    user_prompt = USER_PROMPT_TEMPLATE.format(generated_output=generated_output)
    raw = None
    try:
        raw = call_openrouter(model, SYSTEM_PROMPT, user_prompt)
        cleaned = strip_code_fences(raw)
        parsed = json.loads(cleaned)
        checks = recompute_checks(parsed)
        return {
            "status": "ok",
            "raw_response": raw,
            "parsed_json": json.dumps(parsed, ensure_ascii=False),
            "error": None,
            **checks,
        }
    except Exception as e:
        return {
            "status": "error",
            "raw_response": raw,
            "parsed_json": None,
            "error": f"{type(e).__name__}: {e}",
            "recomputed_formatting_checks_passed": None,
            "recomputed_formatting_score": None,
            "recomputed_readability_ratio": None,
            "recomputed_readability_score": None,
            "validation_issues": [str(e)],
            "is_consistent": False,
        }

In [ ]:
# ## 5. Resume support — load existing output dataset (if any) and skip done work
def load_existing_results():
    try:
        existing = load_dataset(OUTPUT_DATASET_REPO, split="train", token=HF_TOKEN)
        results = existing.to_list()
        done_keys = {(r["item_id"], r["system"], r["judge_model"]) for r in results}
        print(f"Resuming: found {len(results)} existing results "
              f"({len(done_keys)} unique combos) in {OUTPUT_DATASET_REPO}.")
        return results, done_keys
    except Exception as e:
        print(f"No existing output dataset found (starting fresh): {e}")
        return [], set()


all_results, done_keys = load_existing_results()

# ## 6. Build the full work queue: item x judge_model (for SYSTEM_LABEL)
work_items = []
for idx, row in enumerate(input_ds):
    item_id = row.get(ID_COLUMN, idx) if ID_COLUMN in input_ds.column_names else idx
    generated_output = row.get(OUTPUT_COLUMN)
    if generated_output is None:
        continue
    for model in JUDGE_MODELS:
        key = (item_id, SYSTEM_LABEL, model)
        if key in done_keys:
            continue
        work_items.append({
            "item_id": item_id,
            "system": SYSTEM_LABEL,
            "judge_model": model,
            "generated_output": generated_output,
        })

print(f"Total work items remaining: {len(work_items)}")

# ## 7. Checkpoint save helper
def save_checkpoint(results):
    if not results:
        return
    ds = Dataset.from_list(results)
    ds.push_to_hub(OUTPUT_DATASET_REPO, token=HF_TOKEN)
    print(f"  [checkpoint] pushed {len(results)} total results to {OUTPUT_DATASET_REPO}")

# ## 8. Main loop — judges every work item, checkpointing every SAVE_EVERY new results
new_since_last_save = 0

try:
    for i, item in enumerate(work_items, start=1):
        print(f"[{i}/{len(work_items)}] item_id={item['item_id']} "
              f"system={item['system']} model={item['judge_model']}")

        result = judge_one(item["judge_model"], item["generated_output"])
        row = {
            "item_id": item["item_id"],
            "system": item["system"],
            "judge_model": item["judge_model"],
            "timestamp": datetime.now(timezone.utc).isoformat(),
            **result,
            "validation_issues": json.dumps(result["validation_issues"]),
        }
        all_results.append(row)
        new_since_last_save += 1

        if new_since_last_save >= SAVE_EVERY:
            save_checkpoint(all_results)
            new_since_last_save = 0

except KeyboardInterrupt:
    print("Interrupted — saving progress before exiting.")
except Exception:
    print("Unexpected error — saving progress before re-raising.")
    print(traceback.format_exc())
finally:
    if new_since_last_save > 0:
        save_checkpoint(all_results)

print(f"Done. {len(all_results)} total results saved to {OUTPUT_DATASET_REPO}.")


# ## 9. Quick sanity check on judge reliability
inconsistent = [r for r in all_results if not r.get("is_consistent", True)]
print(f"{len(inconsistent)}/{len(all_results)} results had checklist-vs-score mismatches "
      f"(recomputed by this script, per the prompt's own validation note).")
if inconsistent:
    print("Example issues from the first few:")
    for r in inconsistent[:5]:
        print(f" - item_id={r['item_id']} system={r['system']} model={r['judge_model']}: "
              f"{r['validation_issues']}")